# 25 -- Striatum centroid audit in the RAS-resampled frame (2026-09-11)

A second Opus review of `notebooks/24_slab2d_architecture_diversity.ipynb`'s
negative gate result found the real root cause was not the geometry-
rounding bug fixed earlier that day, but that `config.CROP_CENTER_MM` is a
**population-median** offset -- the median row of
`notebooks/01_eda_volumes.ipynb` section 6a's per-volume striatum-centroid-
offset table, whose z component has **sd ~32.5mm** (5-95th percentile
range roughly -97mm to +4mm) in that notebook's measurement. A 12mm-thick
slab fixed at that one point contains real striatal tissue for only
~14-47% of subjects by a normal approximation; the 3D track's 108mm-thick
crop tolerates the same offset error by sheer thickness (~89% by the same
approximation) -- explaining why `slab2d` failed but the 3D CNN (0.4520
solo log loss) clearly works.

That estimate has two real caveats worth resolving before trusting it:
1. Section 6a computed offsets on **raw array axes** (`img.get_fdata()`,
   no resampling), while `crop_or_pad` applies `CROP_CENTER_MM` in the
   **RAS-resampled** frame `resample_to_spacing` produces. For the 40% of
   volumes with oblique affines (README.md), those are different
   coordinate frames -- raw-axis sd may overstate the true anatomical
   spread.
2. Section 6a sampled only 119 of the 1362 training volumes.

This notebook redoes the measurement correctly: `features.striatum_mask`
applied to the **RAS-resampled** volume (the exact frame `crop_or_pad`
uses, the same resampling `load_volume` itself performs) for **all 1362**
volumes, reporting only aggregate statistics -- never per-row/per-uid
values, per the AI-assistant data rule.

**Two things this answers**:
- Is the sd really ~32.5mm in the correct frame, or was the raw-axis
  measurement inflated by oblique-affine frame mismatch?
- **Higher-value question**: does the *production* 3D crop
  (`config.TARGET_SHAPE`/`CROP_SIZE_MM`/`CROP_CENTER_MM`, the crop the
  shipped 0.4185 recipe's CNN actually trains and scores on) fully contain
  the striatum for every subject, or does a meaningful fraction have the
  striatum at or beyond its edge? If the latter, per-subject centering is
  a genuine, previously-unidentified discrimination lever on the
  **production** model -- a bigger prize than `slab2d` ever was.

**Data handling**: loads real `.nii.gz` volumes (resampled, never cropped
to a fixed shape here -- the whole point is measuring where the striatum
actually sits before any fixed-crop assumption is applied) -- per the
AI-assistant data rule, this is **[RUN ME]**: run it yourself, share back
only the printed aggregate numbers. CPU-only, no GPU. Expect roughly the
same per-volume cost as building the 3D `volume_cache` from scratch
(~1.7s/volume per the project's own smoke-test benchmark) since the
dominant cost (`resample_to_spacing`) is the same operation -- so budget
~35-45 minutes for all 1362 volumes. `op02audit` checkpoints its
aggregate arrays (never per-uid identity beyond the uid list itself, kept
only to validate a checkpoint still matches the current label file) to
`data/processed/nb25_op02audit_checkpoint.npz` every 50 volumes, so a
kernel restart or lost session resumes from the last checkpoint instead
of repeating the full run -- not reused by training beyond this
notebook.</cell>


In [ ]:
# [RUN ME] -- loads and resamples real volumes (no crop_or_pad -- this
# measures where the striatum actually sits, not where a fixed crop
# assumes it sits). CPU-only. ~35-45 min for all 1362 volumes.
# Checkpoints to data/processed/nb25_op02audit_checkpoint.npz every 50
# volumes (and reloads it on start) so a kernel restart / closed session
# resumes instead of repeating the full run.
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import nibabel as nib
import numpy as np
import pandas as pd

import config
import data
import features


def _fmt(seconds):
    return str(timedelta(seconds=int(seconds)))


labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
uids = labels_df[config.UID_COLUMN].tolist()
n_total = len(uids)

checkpoint_path = config.DATA_PROCESSED / "nb25_op02audit_checkpoint.npz"

offsets_mm = []  # (x, y, z) centroid offset from geometric center, mm
bbox_min_mm = []  # per-axis min of the striatum mask's bbox, mm offset
bbox_max_mm = []  # per-axis max of the striatum mask's bbox, mm offset
n_degenerate = 0
start_i = 0

if checkpoint_path.exists():
    ckpt = np.load(checkpoint_path, allow_pickle=True)
    if list(ckpt["uids"]) == uids:
        start_i = int(ckpt["next_i"])
        n_degenerate = int(ckpt["n_degenerate"])
        offsets_mm = list(ckpt["offsets_mm"])
        bbox_min_mm = list(ckpt["bbox_min_mm"])
        bbox_max_mm = list(ckpt["bbox_max_mm"])
        print(f"resuming from checkpoint: {start_i}/{n_total} volumes already done "
              f"({n_degenerate} degenerate so far) -- {checkpoint_path}", flush=True)
    else:
        print("checkpoint found but its uid list doesn't match "
              "config.TRAIN_LABELS_PATH -- ignoring it and starting from scratch",
              flush=True)


def _save_checkpoint(next_i):
    np.savez(
        checkpoint_path,
        next_i=next_i,
        n_degenerate=n_degenerate,
        offsets_mm=np.array(offsets_mm) if offsets_mm else np.empty((0, 3)),
        bbox_min_mm=np.array(bbox_min_mm) if bbox_min_mm else np.empty((0, 3)),
        bbox_max_mm=np.array(bbox_max_mm) if bbox_max_mm else np.empty((0, 3)),
        uids=np.array(uids, dtype=object),
    )


start = time.time()
print(f"[{datetime.now():%H:%M:%S}] starting at {start_i}/{n_total} -- progress "
      f"printed and checkpointed every 50 volumes (~30-60s apart)", flush=True)

for i in range(start_i, n_total):
    uid = uids[i]
    path = config.NIFTI_DIR / f"{uid}.nii.gz"
    img = nib.load(str(path))
    resampled, _ = data.resample_to_spacing(img.get_fdata(), img.affine, config.TARGET_SPACING)

    mask = features.striatum_mask(resampled, config.TARGET_SPACING)
    if mask is None or not mask.any():
        n_degenerate += 1
    else:
        idx = np.argwhere(mask)
        shape = np.asarray(resampled.shape, dtype=float)
        center_vox = (shape - 1) / 2.0
        spacing = np.asarray(config.TARGET_SPACING)

        centroid_vox = idx.mean(axis=0)
        offsets_mm.append((centroid_vox - center_vox) * spacing)
        bbox_min_mm.append((idx.min(axis=0) - center_vox) * spacing)
        bbox_max_mm.append((idx.max(axis=0) - center_vox) * spacing)

    if (i + 1) % 50 == 0 or (i + 1) == n_total:
        _save_checkpoint(next_i=i + 1)
        elapsed = time.time() - start
        rate = elapsed / (i + 1 - start_i)
        remaining = rate * (n_total - i - 1)
        eta_clock = datetime.now() + timedelta(seconds=remaining)
        pct = (i + 1) / n_total
        print(f"  [{datetime.now():%H:%M:%S}] {i + 1}/{n_total} ({pct:.0%}) processed, "
              f"elapsed {_fmt(elapsed)}, {rate:.2f}s/volume, "
              f"ETA {_fmt(remaining)} (finish ~{eta_clock:%H:%M:%S}), "
              f"{n_degenerate} degenerate so far, checkpointed", flush=True)

offsets_mm = np.array(offsets_mm)
bbox_min_mm = np.array(bbox_min_mm)
bbox_max_mm = np.array(bbox_max_mm)
print(f"\ndone in {_fmt(time.time() - start)}. "
      f"{len(offsets_mm)}/{n_total} volumes had a usable striatum mask "
      f"({n_degenerate} degenerate/skipped -- for comparison, "
      f"features.py/README.md report 0/1362 skipped for the classical "
      f"baseline's own use of this same mask, so a nonzero count here is "
      f"itself worth a second look before trusting the rest)")


In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Step 1 of the 4th Opus review's diagnostic plan (finding 7): stop here
# before trusting offsets_mm/op03/op04 below. features.striatum_mask keeps
# the 2 largest components in the central 70% of the FOV, which for the
# 630mm large-FOV families extends into the neck -- I-123 ioflupane shows
# thyroid/salivary uptake with incomplete blocking. If the mask's "second
# lobe" is actually a thyroid hot spot rather than the contralateral
# striatum, the measured centroid is dragged ~100-150mm off (consistent
# with the observed left-skew: mean z=-49.5mm vs median z=-38.1mm,
# 5th-pct -138mm). Clean bilateral striatum mask: z-extent ~30-45mm.
# Striatum+thyroid contamination: z-extent ~130mm+.
z_extent_mm = bbox_max_mm[:, 2] - bbox_min_mm[:, 2]
pct = np.percentile(z_extent_mm, [5, 50, 95])
print(f"bbox z-extent (S-I): mean={z_extent_mm.mean():.1f}mm  sd={z_extent_mm.std(ddof=1):.1f}mm  "
      f"5/50/95th pct = {pct[0]:.1f} / {pct[1]:.1f} / {pct[2]:.1f} mm  (n={len(z_extent_mm)})")

n_contaminated = int((z_extent_mm > 100).sum())
print(f"volumes with z-extent > 100mm (likely thyroid/salivary contamination): "
      f"{n_contaminated}/{len(z_extent_mm)} ({n_contaminated / len(z_extent_mm):.1%})")

if pct[1] > 80 or pct[2] > 130:
    print("\n*** STOP: z-extent looks contaminated (finding 7) -- offsets_mm below "
          "are likely NOT a clean striatum-only measurement. Do not proceed to trusting "
          "op03/op04's conclusions until this is investigated (e.g. restrict "
          "striatum_mask's search region away from the neck, or visually inspect a "
          "contaminated case) -- the coverage-gap finding could be an artifact of this. ***")
else:
    print("\nz-extent looks clean -- consistent with a bilateral striatum mask, not "
          "thyroid/salivary contamination. Safe to trust offsets_mm/op03/op04 below.")


In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Answers question 1: is the RAS-resampled-frame sd close to section 6a's
# raw-axis sd (~32.5mm on z, 119-volume sample), or does it differ enough
# to matter -- i.e. was the raw-axis measurement frame-inflated?
axis_names = ["x (L-R)", "y (A-P)", "z (S-I)"]
for a, name in enumerate(axis_names):
    vals = offsets_mm[:, a]
    pct = np.percentile(vals, [5, 50, 95])
    print(f"offset_{name}: mean={vals.mean():+.1f}mm  sd={vals.std(ddof=1):.1f}mm  "
          f"5/50/95th pct = {pct[0]:+.1f} / {pct[1]:+.1f} / {pct[2]:+.1f} mm  "
          f"(n={len(vals)})")

print(f"\nconfig.CROP_CENTER_MM = {config.CROP_CENTER_MM} "
      "(should be close to each axis's median above, by construction -- "
      "notebooks/01 section 6a picked the median as the fixed offset)")
print("for comparison -- notebooks/01 section 6a (raw-axis frame, n=119): "
      "z sd~32.5mm, 5/50/95th pct ~ -96.9 / -15.5 / +4.0 mm")

In [ ]:
# [RUN ME] (no new data access -- uses the arrays built above).
# Answers question 2 (the higher-value one): does the PRODUCTION 3D crop
# (config.TARGET_SHAPE / CROP_SIZE_MM / CROP_CENTER_MM -- the crop the
# shipped 0.4185 recipe's CNN actually trains and scores on) fully contain
# the striatum for every subject?
half_size = np.asarray(config.CROP_SIZE_MM) / 2.0
crop_lo = np.asarray(config.CROP_CENTER_MM) - half_size
crop_hi = np.asarray(config.CROP_CENTER_MM) + half_size
print(f"production 3D crop window (mm offset from geometric center): "
      f"{list(zip(crop_lo.round(1), crop_hi.round(1)))}")

centroid_inside = np.all((offsets_mm >= crop_lo) & (offsets_mm <= crop_hi), axis=1)
bbox_fully_inside = np.all((bbox_min_mm >= crop_lo) & (bbox_max_mm <= crop_hi), axis=1)
bbox_any_outside_per_axis = [
    float(np.mean((bbox_min_mm[:, a] < crop_lo[a]) | (bbox_max_mm[:, a] > crop_hi[a])))
    for a in range(3)
]

print(f"\nstriatum CENTROID inside the production crop: "
      f"{centroid_inside.sum()}/{len(centroid_inside)} ({centroid_inside.mean():.1%})")
print(f"striatum full BBOX inside the production crop: "
      f"{bbox_fully_inside.sum()}/{len(bbox_fully_inside)} ({bbox_fully_inside.mean():.1%})")
print(f"fraction with bbox extending outside the crop, per axis: "
      f"x={bbox_any_outside_per_axis[0]:.1%}  y={bbox_any_outside_per_axis[1]:.1%}  "
      f"z={bbox_any_outside_per_axis[2]:.1%}")
print("\nfor context -- the shipped CNN alone (rung 3, README.md) scores "
      "log loss 0.4520 despite whatever coverage gap this shows, so this "
      "does not mean the production model is broken -- it quantifies "
      "*how much headroom* a per-subject-centered crop could plausibly "
      "recover, not a bug.")

**What we're looking for:** (1) whether the RAS-resampled-frame striatum-
offset spread is close to `notebooks/01` section 6a's raw-axis estimate
(sd~32.5mm on z, n=119) or meaningfully different (oblique-affine frame
mismatch was a real, unresolved caveat on that earlier number); (2) the
higher-value question -- does the *production* 3D crop actually contain
the striatum for (nearly) every subject, or is there a real, previously-
unmeasured coverage gap that per-subject centering could close?

**What we found:**

`op02audit`/`cell-2` (n=1362, 0 degenerate masks): bbox z-extent mean
33.5mm, sd 17.9mm, 5/50/95th pct = 22.1/27.1/86.1mm -- clean bilateral
striatum mask, no thyroid/salivary contamination (only 26/1362, 1.9%,
exceed the 100mm contamination heuristic, well under the STOP threshold).

`op03offsets` (RAS-resampled frame, n=1362):
- offset_x (L-R): mean +3.9mm sd 12.9mm, 5/50/95th pct -12.8/+2.3/+25.0mm
- offset_y (A-P): mean +23.6mm sd 17.4mm, 5/50/95th pct -2.7/+22.4/+54.5mm
- offset_z (S-I): mean -49.5mm sd 41.9mm, 5/50/95th pct -138.0/-38.1/+2.9mm
- `config.CROP_CENTER_MM = (1.5, 22.8, -15.5)`

Question 1 answer: **no**, the RAS-resampled sd is *larger* than the
raw-axis estimate (41.9mm vs ~32.5mm on z), not smaller -- the opposite
of the "raw-axis was frame-inflated" hypothesis. More importantly, the
correct-frame z **median** (-38.1mm) is ~22.6mm away from
`CROP_CENTER_MM`'s z (-15.5mm) -- which instead matches section 6a's
raw-axis median (-15.5mm) exactly. The production crop center was
calibrated against the wrong-frame measurement.

`op04productioncoverage`: production crop window (mm offset) =
x[-66.0,+69.0], y[-12.2,+57.8], z[-68.0,+37.0].
- striatum centroid inside crop: **956/1362 (70.2%)** -- 29.8% of
  subjects have their striatum centroid entirely outside the production
  crop.
- striatum full bbox inside crop: 586/1362 (43.0%)
- fraction with bbox extending outside, per axis: x=10.9% y=34.1% z=36.9%

Question 2 answer: **yes**, a large, real coverage gap exists -- far
worse than the ~89% coverage the intro's normal-approximation predicted.

**Decision / next step:** the coverage gap is real and substantial
(29.8% by centroid). Proceed to `op06coverageloss`'s causal control
before concluding this is a genuine discrimination lever -- confirm the
CNN (not the crop-immune classical baseline) is the one degrading on the
uncovered subgroup.</cell>

In [ ]:
# [RUN ME] (no new volume access -- loads existing OOF prediction arrays
# already on disk, plus reuses op02audit/op04productioncoverage's
# offsets_mm/centroid_inside from this same kernel session).
# Step 2 of the 4th Opus review's diagnostic plan (recommendation #2):
# split the shipped recipe's own honest OOF arrays into covered/uncovered
# subgroups by op04's headline stat (striatum CENTROID inside the
# production crop -- not the bbox-outside numbers, which Opus's review
# found mostly measure mask extent via CROP_SIZE_MM's own back-solved
# construction) and compare CNN-alone vs. classical-baseline degradation.
# This is finding 6's causal control: features.extract_baseline_features
# is CROP-IMMUNE (it runs striatum_mask on the uncropped volume), so if
# only the CNN degrades on the "uncovered" subgroup, the fixed crop is the
# cause; if both degrade about equally, "uncovered" is largely a proxy for
# "harder subject" and per-subject centering would not be expected to help.
import model

repeat_seeds = list(range(config.SEED, config.SEED + 5))
VARIANT_PREFIXES = model.PRODUCTION_VARIANT_PREFIXES  # the 6 shipped variants (no denoise)

EPS = 1e-6


def _to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p))


def _row_log_loss(y, p):
    p = np.clip(p, EPS, 1 - EPS)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


cnn_arrays = [
    np.load(config.DATA_PROCESSED / f"{prefix}_oof_seed{s}.npy")
    for prefix in VARIANT_PREFIXES for s in repeat_seeds
]
baseline_arrays = [
    np.load(config.DATA_PROCESSED / f"baseline_oof_seed{s}.npy") for s in repeat_seeds
]

cnn_pooled = 1.0 / (1.0 + np.exp(-np.mean([_to_logit(a) for a in cnn_arrays], axis=0)))
baseline_pooled = np.mean(baseline_arrays, axis=0)
y_row = labels_df[config.TARGET_COLUMN].to_numpy()

# offsets_mm/centroid_inside were appended only for volumes with a usable
# striatum mask (op02audit skips degenerate ones without recording a uid) --
# if n_degenerate was 0 this is a no-op safety check, but if it wasn't,
# STOP: centroid_inside is silently misaligned with y_row/cnn_pooled by
# row position, and everything below would be wrong.
assert n_degenerate == 0 and len(offsets_mm) == len(y_row) == len(cnn_pooled) == len(baseline_pooled), (
    f"row-count mismatch (offsets_mm={len(offsets_mm)}, y_row={len(y_row)}, "
    f"cnn_pooled={len(cnn_pooled)}, n_degenerate={n_degenerate}) -- centroid_inside "
    "is not indexed by uid, so it cannot be safely aligned with the OOF arrays "
    "when volumes were skipped. Re-derive a per-uid coverage flag before proceeding."
)

cnn_loss = _row_log_loss(y_row, cnn_pooled)
baseline_loss = _row_log_loss(y_row, baseline_pooled)


def _bootstrap_group_delta(loss, covered, n_boot=2000, seed=config.RANDOM_STATE):
    """Unpaired bootstrap (different subjects in each group, not a paired
    candidate-vs-reference comparison like evaluate.paired_bootstrap_ci) on
    the uncovered-minus-covered difference in mean per-row log loss."""
    rng = np.random.default_rng(seed)
    idx_cov = np.flatnonzero(covered)
    idx_unc = np.flatnonzero(~covered)
    point = loss[idx_unc].mean() - loss[idx_cov].mean()
    deltas = np.empty(n_boot)
    for b in range(n_boot):
        s_cov = loss[rng.choice(idx_cov, size=len(idx_cov), replace=True)]
        s_unc = loss[rng.choice(idx_unc, size=len(idx_unc), replace=True)]
        deltas[b] = s_unc.mean() - s_cov.mean()
    return point, np.percentile(deltas, [2.5, 97.5])


n_cov, n_unc = int(centroid_inside.sum()), int((~centroid_inside).sum())
print(f"groups (by op04's centroid-inside-crop flag): covered n={n_cov}  uncovered n={n_unc}\n")

for name, loss in [("CNN (6-variant, logit-pooled -- matches the shipped recipe's CNN side)", cnn_loss),
                    ("classical baseline (crop-immune control)", baseline_loss)]:
    point, (lo, hi) = _bootstrap_group_delta(loss, centroid_inside)
    print(f"{name}:")
    print(f"  covered   mean log loss = {loss[centroid_inside].mean():.4f}")
    print(f"  uncovered mean log loss = {loss[~centroid_inside].mean():.4f}")
    print(f"  uncovered - covered delta = {point:+.4f}   95% bootstrap CI = [{lo:+.4f}, {hi:+.4f}]\n")

print("how to read this: Opus's headline estimate was -0.015 to -0.035 CNN-alone "
      "degradation from poor coverage. If the CNN delta above is clearly positive "
      "(uncovered rows score worse) with a CI excluding 0, and the baseline's delta "
      "is much smaller or crosses 0, that confirms the crop itself (not just subject "
      "difficulty) is the cause -- the causal mechanism finding 6 asked for. If both "
      "deltas are similar in size, \"uncovered\" is mostly a proxy for \"harder subject\" "
      "and per-subject centering is unlikely to recover much.")


**What we're looking for:** op06coverageloss's causal control (finding 6
of the 4th Opus review) -- does the CNN degrade specifically on subjects
whose striatum centroid falls outside the production crop, while the
crop-immune classical baseline does not? That pattern would confirm the
fixed crop itself (not just "these are harder subjects") is costing real
score, and would size the headroom a per-subject-centered crop could
plausibly recover.

**What we found:**

groups (by op04's centroid-inside-crop flag): covered n=956, uncovered
n=406.

CNN (6-variant, logit-pooled -- matches the shipped recipe's CNN side):
- covered mean log loss = 0.3242
- uncovered mean log loss = 0.5558
- uncovered - covered delta = **+0.2317**, 95% bootstrap CI = [+0.1701, +0.2927]

classical baseline (crop-immune control):
- covered mean log loss = 0.5272
- uncovered mean log loss = 0.5292
- uncovered - covered delta = +0.0020, 95% bootstrap CI = [-0.0669, +0.0701]

**Decision / next step:** confirmed. The CNN delta is large, positive,
and its CI excludes 0 by a wide margin; the baseline's delta is
essentially zero and its CI straddles 0. This isolates the fixed crop
(not general subject difficulty) as the cause -- the same subjects show
no meaningful degradation on the crop-immune baseline. The effect size
(+0.2317 on the 29.8% uncovered subgroup) is far larger than Opus's
headline estimate of -0.015 to -0.035 CNN-alone degradation -- this is
not a marginal effect.

Proceed to step 3 of the plan: gate the coverage flag as a 4th blend
covariate (`[logit(cnn), logit(baseline), covered, covered*logit(cnn)]`)
via `evaluate.oof_predict`/`paired_gate`, `min_effect=0.003` -- no
retraining needed, lowest-risk way to spend the last submission if it
clears.</cell>

In [ ]:
# [RUN ME] (no new data access -- reuses cnn_pooled, baseline_pooled, y_row,
# centroid_inside from op06coverageloss's kernel state).
# Step 3 of the 4th Opus review's diagnostic plan: gate the coverage flag
# as a 4th blend covariate via evaluate.oof_predict/paired_gate -- no
# retraining needed. Unlike notebooks/23's three literature candidates
# (all recombinations of information the blend already had access to),
# "is this subject's striatum centroid inside the production crop" is
# genuinely new information the shipped 2-feature blend has never seen.
import evaluate

features_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
family_df = labels_df[[config.UID_COLUMN]].merge(features_df, on=config.UID_COLUMN, how="left")
assert family_df[config.UID_COLUMN].tolist() == labels_df[config.UID_COLUMN].tolist(), (
    "merge reordered or dropped rows -- family would misalign with y_row/"
    "cnn_pooled/centroid_inside, which are all indexed by labels_df's original "
    "row order"
)
family = family_df["inplane_family"].to_numpy()

FOLDS = evaluate.make_folds(y_row, family, n_splits=config.N_FOLDS, random_state=config.RANDOM_STATE)

covered = centroid_inside.astype(float)
cnn_logit = _to_logit(cnn_pooled)
baseline_logit = _to_logit(baseline_pooled)

reference_features = np.column_stack([cnn_logit, baseline_logit])
candidate_features = np.column_stack([cnn_logit, baseline_logit, covered, covered * cnn_logit])

# Reference is refit (not the frozen shipped a/b/c) so the comparison is
# apples-to-apples on the exact same fold split -- matches how notebook 23
# gated its own three candidates against the shipped recipe.
reference_oof = evaluate.oof_predict(reference_features, y_row, FOLDS)
candidate_oof = evaluate.oof_predict(candidate_features, y_row, FOLDS)

result = evaluate.paired_gate(
    "coverage flag as 4th blend covariate (op07coverageblend)",
    candidate_oof, reference_oof, y_row, min_effect=0.003,
)


**What we're looking for:** does telling the blend "this subject's
striatum centroid falls outside the production crop" (as a 4th covariate,
plus its interaction with the CNN logit) let it discount the CNN more on
exactly the subjects where op06coverageloss showed it degrades -- without
retraining anything? Per the plan (roadmap step 3), this is the
lowest-risk way to spend the last submission if it clears the same
`min_effect=0.003` / whole-CI-negative bar used throughout this project.

**What we found:**

```
candidate row-CV log loss = 0.3564   reference = 0.3617
delta (candidate - reference) = -0.0053   95% paired-bootstrap CI = [-0.0110, +0.0001]
-> does NOT clear the gate (rule: whole CI < 0 AND |delta| > 0.003)
```

**Decision:** does not clear, per the pre-registered rule, exactly as
written before running it -- the CI's upper bound (+0.0001) is a hair
above zero. Honest framing: this is a near-miss, not a clean pass or a
clean null. It's a materially stronger result than notebook 23's three
literature candidates (point deltas -0.0023 to -0.0031, CIs like
[-0.0092, +0.0027] -- much wider, further from clearing) -- the coverage
flag really does carry information the shipped blend doesn't have. But
"almost clears" is still "does not clear": re-running with a larger
`n_bootstrap` or a different seed purely to see if the interval tips
negative would be exactly the post-hoc gate-gaming this project has
avoided everywhere else (flip-TTA, denoise, the 3 literature candidates
were all held to the same bar without a second look once measured). The
result stands as-is: **do not spend the last submission on this
covariate alone.**

**Why it's informative anyway:** a 2-parameter logistic reweighting of
already-existing predictions came this close. Step 2 measured the
underlying degradation on uncovered subjects at +0.2317 log loss (406
subjects) -- a blend covariate can only ever partially compensate for
that (it can down-weight a bad CNN prediction, it can't recover the
anatomy the crop never saw). That ceiling argument, plus this near-miss,
both point the same direction: **step 4 (fixing the crop itself, not
just discounting its output) is the higher-value remaining lever**, not
a smaller variant of step 3.

**Next step:** proceed to step 4 of the plan -- start with the cheap
`CROP_CENTER_MM=(2.3, 22.4, -38.1)` re-centering control (no per-subject
logic, one constant change) before full per-subject centering, retrain
the rung3 variant only in a separate cache dir, gate against
`rung3_oof_seed{42-46}.npy`.

In [ ]:
# [RUN ME] (no new data access -- reuses offsets_mm/bbox_min_mm/bbox_max_mm
# from op02audit and crop_lo/crop_hi/centroid_inside from op04productioncoverage).
# Phase A, item 1 of the 5th Opus review: op04 computed the production crop
# window as CROP_CENTER_MM +/- CROP_SIZE_MM/2, but crop_or_pad (src/data.py)
# never uses CROP_SIZE_MM -- it derives the window from TARGET_SHAPE voxels
# at TARGET_SPACING. The two differ by ~1.4-1.9mm per face (CROP_SIZE_MM was
# itself back-solved from a mask-bbox percentile, not from the crop geometry
# crop_or_pad actually applies). Doesn't reverse any conclusion (true
# coverage should be a touch above op04's 70.2%), but it mislabels the
# boundary subjects that op07 just gated -- fix before trusting coverage
# numbers used to justify GPU time. (Not modeling crop_or_pad's additional
# +/-1-voxel rounding wrinkle from each volume's own resampled-shape parity
# -- Opus flagged that as second-order/~1mm, and this notebook doesn't have
# each volume's exact resampled shape on hand to reproduce it exactly.)
half_size_fixed = (np.asarray(config.TARGET_SHAPE) * np.asarray(config.TARGET_SPACING)) / 2.0
crop_lo_fixed = np.asarray(config.CROP_CENTER_MM) - half_size_fixed
crop_hi_fixed = np.asarray(config.CROP_CENTER_MM) + half_size_fixed
print(f"corrected production crop window (mm offset from geometric center): "
      f"{list(zip(crop_lo_fixed.round(1), crop_hi_fixed.round(1)))}")
print(f"op04's window (CROP_SIZE_MM-based, WRONG): "
      f"{list(zip(crop_lo.round(1), crop_hi.round(1)))}")

centroid_inside_fixed = np.all((offsets_mm >= crop_lo_fixed) & (offsets_mm <= crop_hi_fixed), axis=1)
n_relabeled = int((centroid_inside_fixed != centroid_inside).sum())
print(f"\nstriatum centroid inside CORRECTED production crop: "
      f"{centroid_inside_fixed.sum()}/{len(centroid_inside_fixed)} ({centroid_inside_fixed.mean():.1%}) "
      f"(op04's buggy window said {centroid_inside.mean():.1%})")
print(f"{n_relabeled}/{len(centroid_inside)} subjects change covered/uncovered label from the fix "
      f"({n_relabeled / len(centroid_inside):.1%})")

# Hypothetical coverage under the two step-4 candidate fixes, on the SAME
# corrected window geometry -- answers "how much would either fix actually
# change the denominator" before spending any GPU time on either.
recentered_offset = np.array([2.3, 22.4, -38.1])  # op03's corrected median (x, y, z)
crop_lo_recentered = recentered_offset - half_size_fixed
crop_hi_recentered = recentered_offset + half_size_fixed
centroid_inside_recentered = np.all(
    (offsets_mm >= crop_lo_recentered) & (offsets_mm <= crop_hi_recentered), axis=1
)
print(f"\nhypothetical coverage if CROP_CENTER_MM were re-centered to {tuple(recentered_offset)}: "
      f"{centroid_inside_recentered.sum()}/{len(centroid_inside_recentered)} "
      f"({centroid_inside_recentered.mean():.1%})")
print("hypothetical coverage under full per-subject centering: 100.0% by construction "
      "(the crop would be centered on each subject's own measured centroid)")


In [ ]:
# [RUN ME] (no new data access -- reuses family/cnn_loss/baseline_loss from
# op06coverageloss and centroid_inside_fixed from op08fixcoverage).
# Phase A, item 2 of the 5th Opus review (finding 7): op06's causal control
# assumed "uncovered" isn't just a proxy for a family/FOV confound (the
# classical baseline is ComBat-harmonized BY inplane_family, the CNN is
# not -- if "uncovered" tracks family, the baseline's flat delta is partly
# guaranteed by construction, not purely evidence of crop-immunity). Two
# free checks settle it: does "uncovered" cut across families, and does the
# CNN's gap survive within a single family?
family_series = pd.Series(family)
covered_label = pd.Series(np.where(centroid_inside_fixed, "covered", "uncovered"))
crosstab = pd.crosstab(family_series, covered_label)
crosstab["pct_uncovered"] = crosstab["uncovered"] / (crosstab["covered"] + crosstab["uncovered"])
print("coverage by inplane_family (does 'uncovered' concentrate in 1-2 families, or cut across them?):")
print(crosstab.sort_values("pct_uncovered", ascending=False))

eligible = crosstab[(crosstab["covered"] >= 20) & (crosstab["uncovered"] >= 20)]
if len(eligible) == 0:
    print("\nno family has >=20 subjects in BOTH groups -- within-family check not powered, skip it.")
else:
    sizes = eligible["covered"] + eligible["uncovered"]
    largest_family = sizes.idxmax()
    mask = (family == largest_family)
    print(f"\nwithin-family check, largest family with >=20 in both groups: "
          f"family={largest_family!r}, n={int(mask.sum())} "
          f"(covered={int((mask & centroid_inside_fixed).sum())}, "
          f"uncovered={int((mask & ~centroid_inside_fixed).sum())})")

    point, (lo, hi) = _bootstrap_group_delta(cnn_loss[mask], centroid_inside_fixed[mask])
    print(f"  CNN within-family uncovered-covered delta      = {point:+.4f}   95% CI=[{lo:+.4f}, {hi:+.4f}]")
    point_b, (lo_b, hi_b) = _bootstrap_group_delta(baseline_loss[mask], centroid_inside_fixed[mask])
    print(f"  baseline within-family uncovered-covered delta = {point_b:+.4f}   95% CI=[{lo_b:+.4f}, {hi_b:+.4f}]")
    print("\nif the CNN delta stays large/positive and the baseline delta stays flat WITHIN this "
          "single family, the family confound is ruled out and the crop story is nailed. If the "
          "CNN delta shrinks a lot here, some of op06's +0.2317 was riding on the family/FOV axis.")

print("\nper-group prevalence and AUROC (AUROC is immune to \"the baseline is a weaker model so "
      "its log-loss gap looks smaller\" -- separates discrimination loss from calibration loss):")
for name, mask_grp in [("covered", centroid_inside_fixed), ("uncovered", ~centroid_inside_fixed)]:
    prevalence = y_row[mask_grp].mean()
    cnn_auc = evaluate.auroc_score(y_row[mask_grp], cnn_pooled[mask_grp])
    baseline_auc = evaluate.auroc_score(y_row[mask_grp], baseline_pooled[mask_grp])
    print(f"  {name} (n={int(mask_grp.sum())}): prevalence={prevalence:.3f}  "
          f"CNN AUROC={cnn_auc:.4f}  baseline AUROC={baseline_auc:.4f}")


In [ ]:
# [RUN ME] (no new data access -- reuses candidate_oof/reference_oof from
# op07coverageblend, crop_lo_fixed/crop_hi_fixed from op08fixcoverage).
# Phase A, item 3 of the 5th Opus review, part 1: re-run op07's EXACT same
# candidate (same fitted candidate_oof/reference_oof arrays -- the bootstrap
# only resamples rows of already-computed predictions, it never refits
# anything) through the higher-precision bootstrap (src/evaluate.py's
# n_bootstrap default raised 1000->20000 today, 2026-09-13, specifically
# because op07's CI upper bound (+0.0001) sat inside the old n=1000 Monte
# Carlo noise (~0.00024 SE) -- this is measuring the SAME estimand more
# precisely, not re-running with a different seed to shop for a flip.
print("=== same candidate as op07, re-run ONLY for bootstrap precision (not a new test) ===")
result_highprecision = evaluate.paired_gate(
    "coverage flag as 4th blend covariate -- SAME as op07, n_bootstrap 1000->20000",
    candidate_oof, reference_oof, y_row, min_effect=0.003,
)


In [ ]:
# [RUN ME] (no new data access -- reuses cnn_logit/baseline_logit/reference_oof
# from op07coverageblend and crop_lo_fixed/crop_hi_fixed from op08fixcoverage).
# Phase A, item 3 of the 5th Opus review, part 2: a SECOND, differently-
# specified candidate (pre-registered here, not a re-run of op07/op10 above)
# -- op07's binary "covered" flag treats a subject 5mm outside the crop the
# same as one 70mm outside (op03's z-offsets span -138mm to +2.9mm at the
# 5th/95th pct against a ~105mm-tall crop). A continuous penetration-depth
# covariate, plus letting it interact with BOTH logits (op07 only let it
# discount the CNN; under the crop-degradation story the blend should also
# be free to up-weight the crop-immune baseline more on badly-covered
# subjects), is strictly more informative and costs nothing new.
penetration_mm = np.maximum(crop_lo_fixed - offsets_mm, 0.0) + np.maximum(offsets_mm - crop_hi_fixed, 0.0)
dist_outside_mm = np.sqrt((penetration_mm ** 2).sum(axis=1))  # 0 for covered subjects
print(f"dist_outside_mm: {(dist_outside_mm > 0).sum()}/{len(dist_outside_mm)} nonzero (uncovered), "
      f"mean over uncovered={dist_outside_mm[dist_outside_mm > 0].mean():.1f}mm, "
      f"max={dist_outside_mm.max():.1f}mm")

candidate_features_v2 = np.column_stack([
    cnn_logit, baseline_logit, dist_outside_mm,
    dist_outside_mm * cnn_logit, dist_outside_mm * baseline_logit,
])
candidate_oof_v2 = evaluate.oof_predict(candidate_features_v2, y_row, FOLDS)

result_v2 = evaluate.paired_gate(
    "continuous coverage-penetration covariate + both-logit interactions "
    "(pre-registered 2nd candidate, not a re-run of op07/op10)",
    candidate_oof_v2, reference_oof, y_row, min_effect=0.003,
)


**What we're looking for (Phase A of the 5th Opus review, before any GPU
time is spent on step 4):** (1) does fixing op04's crop-window bug
materially change the coverage numbers or op06/op07's conclusions; (2) is
op06's causal story (crop, not subject difficulty) confounded by family/FOV
membership (the classical baseline is ComBat-harmonized by
`inplane_family`, the CNN isn't); (3) was op07's near-miss verdict a real
null, or an artifact of `n_bootstrap=1000`'s own Monte Carlo noise, and
does a more informative (continuous, both-logit-interacting) covariate
specification change the answer.

**What we found:**

- **op08 (window bug fix):** corrected window vs. op04's buggy one differs
  by only ~1.0-2.0mm per face; coverage barely moves (70.2% -> 71.1%, only
  13/1362 subjects relabeled). The bug is real but immaterial to every
  conclusion drawn so far.
- **op08 (hypothetical coverage):** re-centering the fixed `CROP_CENTER_MM`
  constant to op03's corrected median (2.3, 22.4, -38.1mm) -> **79.0%**
  coverage (+8 points). Per-subject centering -> **100.0%** by
  construction.
- **op09 (family confound):** real but partial. "Uncovered" concentrates
  almost entirely in 2 of 17 families (2.46mm: 56.6% uncovered; 3.895mm:
  24.9%; every other family <=6%, most exactly 0%) -- these are the same
  two families notebook 07's per-family table already flagged as the
  worst-scoring (0.5132 vs. 0.3617 pooled). But the causal story survives
  **within** the largest single family (2.46mm, n=528: 229 covered / 299
  uncovered): CNN delta = **+0.1758**, 95% CI=[+0.0911, +0.2617] (clearly
  excludes 0), baseline delta = +0.0292, CI=[-0.0533, +0.1106] (crosses
  0). Smaller than op06's pooled +0.2317 (some of that WAS riding on the
  family axis, as Opus's finding 7 suspected) but still large and clean.
- **op09 (AUROC decomposition, the strongest new evidence):** CNN AUROC
  collapses from **0.9293 (covered) to 0.7088 (uncovered)**; baseline
  AUROC only drifts from 0.8112 to 0.7666. AUROC is prevalence-invariant
  (uncovered subjects do have a higher base rate, 0.697 vs. 0.488), so
  this cleanly isolates a CNN-specific discrimination failure on uncovered
  subjects, not a baseline-shared "harder subjects" effect.
- **op10 (same op07 candidate, n_bootstrap 1000->20000):** identical
  result to 4 decimal places (delta=-0.0053, CI=[-0.0110, +0.0001]). This
  was NOT a Monte Carlo artifact -- op07's near-miss is a stable, genuine
  estimate at the boundary, not an under-sampled one.
- **op11 (continuous coverage-penetration covariate + both-logit
  interactions, pre-registered 2nd candidate):** delta=+0.0002,
  CI=[-0.0030, +0.0032] -- flat, actually slightly worse than op07's
  simpler binary covariate. Letting the coverage signal reweight the
  baseline too adds no value (consistent with the baseline being
  structurally crop-immune -- there's no real mechanism for coverage to
  modulate trust in it) and the extra degrees of freedom cost more than
  the finer distance signal buys.

**Decision:** the causal story holds (a real, if smaller-than-pooled, CNN
discrimination failure specific to the fixed crop, not general subject
difficulty), and -- as predicted -- no blend-covariate reweighting
recovers it, because it's a discrimination loss, not a calibration one.
Proceed to Phase B: `notebooks/26_striatum_coverage_retrain.ipynb`
trains a rung3-equivalent CNN under both the re-centered-constant (79%
coverage) and per-subject-centered (100% coverage) crop, gated against
`rung3_oof_seed{42-46}.npy` via `evaluate.paired_repeat_gate`. Decision
deadline (per the review): end of 2026-09-14 -- if neither arm clears
convincingly by then, ship nothing further and keep 0.4185.